# Machine Learning

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import warnings
from scipy.stats import uniform, randint
import sys
import timm
from torch.utils.data import Dataset, random_split

In [2]:
df = pd.read_parquet("../data/processed/anime_data_2.parquet")
df.info()

NameError: name 'pd' is not defined

## Baseline (Random Forest)

For our baseline model, we will use the following data:
* Source (we will use more advanced and engineered features regarding source material later)
* Genres (one hot from feature engineering)
* Themes (one hot from feature engineering)
* Episode Count
* Demographics (one hot from feature engineering)
* Age Rating
* Sequel

Our metrics will be the following (z-scored against cohort):
* Score 
* Watching + Completed 
* Favorites
* Drop Rate
* Forum Posts

In [ ]:
X = df.drop(columns=['mal_id', 'title', 'producers', 'studios', 'demographics', 'genres', 'themes', 'cohort', 'forum_z', 'score_z', 'wc_z', 'favorites_z', 'drop_rate_z'])
X.info()

In [ ]:
X['sequel'] = X['sequel'].astype(int)

X_encoded = pd.get_dummies(X, columns=['rating', 'source'])
y = df['score_z']

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42
)

warnings.filterwarnings("ignore", category=UserWarning)

base_model = RandomForestRegressor(

    n_jobs=-1,

    bootstrap=True,

    random_state=42

)

param_distributions = {

    "n_estimators": [100, 250, 500],

    "max_depth": randint(3, 20),

    "min_samples_split": randint(10, 100),

    "min_samples_leaf": randint(5, 50),

    "max_features": uniform(0.3, 0.7),

    "max_samples": uniform(0.4, 0.4)

}

random_search = RandomizedSearchCV(

    estimator=base_model,

    param_distributions=param_distributions,

    n_iter=20,

    scoring="neg_mean_absolute_error",

    cv=5,

    random_state=42,

    n_jobs=1,

    verbose=3

)

print(base_model.get_params())

random_search.fit(
    X_train_sub,
    y_train_sub,
)

best_params = random_search.best_params_.copy()
best_params.pop("n_estimators", None)

step = 10
patience = 50
max_estimators = 2000

final_model = RandomForestRegressor(
    n_jobs=-1,
    bootstrap=True,
    warm_start=True,
    random_state=42,
    n_estimators=step,
    **best_params
)

best_val_mae = np.inf
best_n_estimators = step
no_improve_trees = 0

while True:
    final_model.fit(X_train_sub, y_train_sub)

    val_preds = final_model.predict(X_val)
    val_mae = mean_absolute_error(y_val, val_preds)

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_n_estimators = final_model.n_estimators
        no_improve_trees = 0
    else:
        no_improve_trees += step

    if no_improve_trees >= patience or final_model.n_estimators >= max_estimators:
        break

    final_model.n_estimators += step

final_model.n_estimators = best_n_estimators
final_model.estimators_ = final_model.estimators_[:best_n_estimators]

print(f"\nStopped at {best_n_estimators} trees (best val MAE: {best_val_mae:.3f})")

best_model = final_model

predictions = best_model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)

rmse = np.sqrt(mean_squared_error(y_test, predictions))

r2 = r2_score(y_test, predictions)

print("\n----- Regression Results -----")

print(f"MAE : {mae:.3f}")

print(f"RMSE: {rmse:.3f}")

print(f"R²  : {r2:.3f}")

print("Best Parameters:", random_search.best_params_)

In [ ]:
print(predictions)
print(X_test)

Now, for wc_z:

In [ ]:
y = df['wc_z']

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42
)

warnings.filterwarnings("ignore", category=UserWarning)

base_model = RandomForestRegressor(

    n_jobs=-1,

    bootstrap=True,

    random_state=42

)

param_distributions = {

    "n_estimators": [100, 250, 500],

    "max_depth": randint(3, 20),

    "min_samples_split": randint(10, 100),

    "min_samples_leaf": randint(5, 50),

    "max_features": uniform(0.3, 0.7),

    "max_samples": uniform(0.4, 0.4)

}

random_search = RandomizedSearchCV(

    estimator=base_model,

    param_distributions=param_distributions,

    n_iter=20,

    scoring="neg_mean_absolute_error",

    cv=5,

    random_state=42,

    n_jobs=1,

    verbose=3

)

print(base_model.get_params())

random_search.fit(
    X_train_sub,
    y_train_sub,
)

best_params = random_search.best_params_.copy()
best_params.pop("n_estimators", None)

step = 10
patience = 50
max_estimators = 2000

final_model = RandomForestRegressor(
    n_jobs=-1,
    bootstrap=True,
    warm_start=True,
    random_state=42,
    n_estimators=step,
    **best_params
)

best_val_mae = np.inf
best_n_estimators = step
no_improve_trees = 0

while True:
    final_model.fit(X_train_sub, y_train_sub)

    val_preds = final_model.predict(X_val)
    val_mae = mean_absolute_error(y_val, val_preds)

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_n_estimators = final_model.n_estimators
        no_improve_trees = 0
    else:
        no_improve_trees += step

    if no_improve_trees >= patience or final_model.n_estimators >= max_estimators:
        break

    final_model.n_estimators += step

final_model.n_estimators = best_n_estimators
final_model.estimators_ = final_model.estimators_[:best_n_estimators]

print(f"\nStopped at {best_n_estimators} trees (best val MAE: {best_val_mae:.3f})")

best_model = final_model

predictions = best_model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)

rmse = np.sqrt(mean_squared_error(y_test, predictions))

r2 = r2_score(y_test, predictions)

print("\n----- Regression Results -----")

print(f"MAE : {mae:.3f}")

print(f"RMSE: {rmse:.3f}")

print(f"R²  : {r2:.3f}")

print("Best Parameters:", random_search.best_params_)

In [ ]:
y = df['favorites_z']

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42
)

warnings.filterwarnings("ignore", category=UserWarning)

base_model = RandomForestRegressor(

    n_jobs=-1,

    bootstrap=True,

    random_state=42

)

param_distributions = {

    "n_estimators": [100, 250, 500],

    "max_depth": randint(3, 20),

    "min_samples_split": randint(10, 100),

    "min_samples_leaf": randint(5, 50),

    "max_features": uniform(0.3, 0.7),

    "max_samples": uniform(0.4, 0.4)

}

random_search = RandomizedSearchCV(

    estimator=base_model,

    param_distributions=param_distributions,

    n_iter=20,

    scoring="neg_mean_absolute_error",

    cv=5,

    random_state=42,

    n_jobs=1,

    verbose=3

)

print(base_model.get_params())

random_search.fit(
    X_train_sub,
    y_train_sub,
)

best_params = random_search.best_params_.copy()
best_params.pop("n_estimators", None)

step = 10
patience = 50
max_estimators = 2000

final_model = RandomForestRegressor(
    n_jobs=-1,
    bootstrap=True,
    warm_start=True,
    random_state=42,
    n_estimators=step,
    **best_params
)

best_val_mae = np.inf
best_n_estimators = step
no_improve_trees = 0

while True:
    final_model.fit(X_train_sub, y_train_sub)

    val_preds = final_model.predict(X_val)
    val_mae = mean_absolute_error(y_val, val_preds)

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_n_estimators = final_model.n_estimators
        no_improve_trees = 0
    else:
        no_improve_trees += step

    if no_improve_trees >= patience or final_model.n_estimators >= max_estimators:
        break

    final_model.n_estimators += step

final_model.n_estimators = best_n_estimators
final_model.estimators_ = final_model.estimators_[:best_n_estimators]

print(f"\nStopped at {best_n_estimators} trees (best val MAE: {best_val_mae:.3f})")

best_model = final_model

predictions = best_model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)

rmse = np.sqrt(mean_squared_error(y_test, predictions))

r2 = r2_score(y_test, predictions)

print("\n----- Regression Results -----")

print(f"MAE : {mae:.3f}")

print(f"RMSE: {rmse:.3f}")

print(f"R²  : {r2:.3f}")

print("Best Parameters:", random_search.best_params_)

In [ ]:
y = df['drop_rate_z']

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42
)

warnings.filterwarnings("ignore", category=UserWarning)

base_model = RandomForestRegressor(

    n_jobs=-1,

    bootstrap=True,

    random_state=42

)

param_distributions = {

    "n_estimators": [100, 250, 500],

    "max_depth": randint(3, 20),

    "min_samples_split": randint(10, 100),

    "min_samples_leaf": randint(5, 50),

    "max_features": uniform(0.3, 0.7),

    "max_samples": uniform(0.4, 0.4)

}

random_search = RandomizedSearchCV(

    estimator=base_model,

    param_distributions=param_distributions,

    n_iter=20,

    scoring="neg_mean_absolute_error",

    cv=5,

    random_state=42,

    n_jobs=1,

    verbose=3

)

print(base_model.get_params())

random_search.fit(
    X_train_sub,
    y_train_sub,
)

best_params = random_search.best_params_.copy()
best_params.pop("n_estimators", None)

step = 10
patience = 50
max_estimators = 2000

final_model = RandomForestRegressor(
    n_jobs=-1,
    bootstrap=True,
    warm_start=True,
    random_state=42,
    n_estimators=step,
    **best_params
)

best_val_mae = np.inf
best_n_estimators = step
no_improve_trees = 0

while True:
    final_model.fit(X_train_sub, y_train_sub)

    val_preds = final_model.predict(X_val)
    val_mae = mean_absolute_error(y_val, val_preds)

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_n_estimators = final_model.n_estimators
        no_improve_trees = 0
    else:
        no_improve_trees += step

    if no_improve_trees >= patience or final_model.n_estimators >= max_estimators:
        break

    final_model.n_estimators += step

final_model.n_estimators = best_n_estimators
final_model.estimators_ = final_model.estimators_[:best_n_estimators]

print(f"\nStopped at {best_n_estimators} trees (best val MAE: {best_val_mae:.3f})")

best_model = final_model

predictions = best_model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)

rmse = np.sqrt(mean_squared_error(y_test, predictions))

r2 = r2_score(y_test, predictions)

print("\n----- Regression Results -----")

print(f"MAE : {mae:.3f}")

print(f"RMSE: {rmse:.3f}")

print(f"R²  : {r2:.3f}")

print("Best Parameters:", random_search.best_params_)

In [ ]:
y = df['forum_z']

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42
)

warnings.filterwarnings("ignore", category=UserWarning)

base_model = RandomForestRegressor(

    n_jobs=-1,

    bootstrap=True,

    random_state=42

)

param_distributions = {

    "n_estimators": [100, 250, 500],

    "max_depth": randint(3, 20),

    "min_samples_split": randint(10, 100),

    "min_samples_leaf": randint(5, 50),

    "max_features": uniform(0.3, 0.7),

    "max_samples": uniform(0.4, 0.4)

}

random_search = RandomizedSearchCV(

    estimator=base_model,

    param_distributions=param_distributions,

    n_iter=20,

    scoring="neg_mean_absolute_error",

    cv=5,

    random_state=42,

    n_jobs=1,

    verbose=3

)

print(base_model.get_params())

random_search.fit(
    X_train_sub,
    y_train_sub,
)

best_params = random_search.best_params_.copy()
best_params.pop("n_estimators", None)

step = 10
patience = 50
max_estimators = 2000

final_model = RandomForestRegressor(
    n_jobs=-1,
    bootstrap=True,
    warm_start=True,
    random_state=42,
    n_estimators=step,
    **best_params
)

best_val_mae = np.inf
best_n_estimators = step
no_improve_trees = 0

while True:
    final_model.fit(X_train_sub, y_train_sub)

    val_preds = final_model.predict(X_val)
    val_mae = mean_absolute_error(y_val, val_preds)

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_n_estimators = final_model.n_estimators
        no_improve_trees = 0
    else:
        no_improve_trees += step

    if no_improve_trees >= patience or final_model.n_estimators >= max_estimators:
        break

    final_model.n_estimators += step

final_model.n_estimators = best_n_estimators
final_model.estimators_ = final_model.estimators_[:best_n_estimators]

print(f"\nStopped at {best_n_estimators} trees (best val MAE: {best_val_mae:.3f})")

best_model = final_model

predictions = best_model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)

rmse = np.sqrt(mean_squared_error(y_test, predictions))

r2 = r2_score(y_test, predictions)

print("\n----- Regression Results -----")

print(f"MAE : {mae:.3f}")

print(f"RMSE: {rmse:.3f}")

print(f"R²  : {r2:.3f}")

print("Best Parameters:", random_search.best_params_)

## Input Preparation

Right now, the priority is to reduce dimensions. The plan is the following:
* Reduce the dimensions of the image tensors from 512
* Reduce the dimensions of sentimental analysis tensors from 768
* Reduce the pool of producers and studios into embedded vectors

### Studios

In [ ]:
le_studios = df['studios'].explode().unique()
studio_to_idx = {s: i for i, s in enumerate(le_studios)}
n_studios = len(le_studios)
embed_dim = 5
le_data = [
    (studios, {"score_z": score_z, "wc_z": wc_z, "favorites_z": favorites_z, "drop_rate_z": drop_rate_z, "forum_z": forum_z})
    for studios, score_z, wc_z, favorites_z, drop_rate_z, forum_z
    in zip(df['studios'], df['score_z'], df['wc_z'], df['favorites_z'], df['drop_rate_z'], df['forum_z'])
]

flat_indices = [] 
offsets = [0] 
for studio_list, _ in le_data: 
    idxs = [studio_to_idx[s] for s in studio_list] 
    flat_indices.extend(idxs) 
    offsets.append(offsets[-1] + len(idxs)) 

offsets = offsets[:-1]

flat_indices = torch.tensor(flat_indices, dtype=torch.long) 
offsets = torch.tensor(offsets, dtype=torch.long)
targets = ["score_z", "wc_z", "favorites_z", "drop_rate_z", "forum_z"]
y = torch.tensor([[d[t] for t in targets] for _, d in le_data], dtype=torch.float32)

class MultiTargetStudioModel(nn.Module): 
    def __init__(self, n_studios, embed_dim, n_targets, hidden_dim=16): 
        super().__init__() 
        self.studio_embed = nn.EmbeddingBag(n_studios, embed_dim, mode="mean") # shared trunk 
        self.trunk = nn.Sequential( nn.Linear(embed_dim, hidden_dim), nn.ReLU(), ) # one head per target -- ModuleList so each gets its own weights 
        self.heads = nn.ModuleList([nn.Linear(hidden_dim, 1) for _ in range(n_targets)])

    def forward(self, flat_indices, offsets): 
        studio_vec = self.studio_embed(flat_indices, offsets) # (batch, embed_dim), pooled 
        shared = self.trunk(studio_vec) # (batch, hidden_dim) 
        outputs = [head(shared) for head in self.heads] # list of (batch, 1) 
        return torch.cat(outputs, dim=1)

model = MultiTargetStudioModel(n_studios, embed_dim, n_targets=len(targets)) 
optimizer = torch.optim.Adam(model.parameters(), lr=0.05) 
loss_fn = nn.MSELoss()

for epoch in range(400): 
    optimizer.zero_grad() 
    pred = model(flat_indices, offsets) 
    loss = loss_fn(pred, y) 
    loss.backward() 
    optimizer.step()

print(f"Final loss: {loss.item():.4f}\n") 
print("Predictions vs actual:") 
# with torch.no_grad(): 
    # pred = model(flat_indices, offsets) 
    # print("pred shape:", pred.shape)
    # print("y shape:", y.shape)
    # print("len(data):", len(le_data))
    # print("offsets:", offsets)
    # print("flat_indices:", flat_indices)
    # for i, (studio_list, _) in enumerate(le_data): 
    #     print(f" {str(studio_list):35s} pred={pred[i].numpy()} actual={y[i].numpy()}")

print("\nLearned studio embeddings:") 
for s, i in studio_to_idx.items():
    print(f" {str(s):15s} {model.studio_embed.weight[i].detach().numpy()}")

with torch.no_grad():
    pred = model(flat_indices, offsets)
    per_target_mse = ((pred - y) ** 2).mean(dim=0)
    for t, mse in zip(targets, per_target_mse):
        print(f"{t:15s} MSE: {mse.item():.4f}")

Two things to note:
1. We can roughly interpret this as: around 41% of overall metrics can be explained by the studio working on it. This is informative and pretty surprising.
2. If we break it down into each metric, we can see that the drop rate has a much higher MSE. This is fair; big studios can flop and make the users drop their shows.

In [ ]:
studios_embedded = model.studio_embed.weight
print(studios_embedded)

### Producers

In [ ]:
le_producers = df['producers'].explode().unique()
producer_to_idx = {s: i for i, s in enumerate(le_producers)}
n_producers = len(le_producers)
embed_dim = 5
le_data = [
    (producers, {"score_z": score_z, "wc_z": wc_z, "favorites_z": favorites_z, "drop_rate_z": drop_rate_z, "forum_z": forum_z})
    for producers, score_z, wc_z, favorites_z, drop_rate_z, forum_z
    in zip(df['producers'], df['score_z'], df['wc_z'], df['favorites_z'], df['drop_rate_z'], df['forum_z'])
]

flat_indices = [] 
offsets = [0] 
for producer_list, _ in le_data: 
    idxs = [producer_to_idx[s] for s in producer_list] 
    flat_indices.extend(idxs) 
    offsets.append(offsets[-1] + len(idxs)) 

offsets = offsets[:-1]

flat_indices = torch.tensor(flat_indices, dtype=torch.long) 
offsets = torch.tensor(offsets, dtype=torch.long)
targets = ["score_z", "wc_z", "favorites_z", "drop_rate_z", "forum_z"]
y = torch.tensor([[d[t] for t in targets] for _, d in le_data], dtype=torch.float32)

model = MultiTargetStudioModel(n_producers, embed_dim, n_targets=len(targets)) 
optimizer = torch.optim.Adam(model.parameters(), lr=0.05) 
loss_fn = nn.MSELoss()

for epoch in range(400): 
    optimizer.zero_grad() 
    pred = model(flat_indices, offsets) 
    loss = loss_fn(pred, y) 
    loss.backward() 
    optimizer.step()

print(f"Final loss: {loss.item():.4f}\n") 
# print("Predictions vs actual:") 
# with torch.no_grad(): 
    # pred = model(flat_indices, offsets) 
    # print("pred shape:", pred.shape)
    # print("y shape:", y.shape)
    # print("len(data):", len(le_data))
    # print("offsets:", offsets)
    # print("flat_indices:", flat_indices)
    # for i, (studio_list, _) in enumerate(le_data): 
    #     print(f" {str(studio_list):35s} pred={pred[i].numpy()} actual={y[i].numpy()}")

print("\nLearned studio embeddings:") 
for s, i in studio_to_idx.items():
    print(f" {str(s):15s} {model.studio_embed.weight[i].detach().numpy()}")

with torch.no_grad():
    pred = model(flat_indices, offsets)
    per_target_mse = ((pred - y) ** 2).mean(dim=0)
    for t, mse in zip(targets, per_target_mse):
        print(f"{t:15s} MSE: {mse.item():.4f}")

### Synopsis

In [ ]:
semantic_embeddings = np.load('../data/processed/semantic_embeddings.npy')
print(semantic_embeddings)
print(type(semantic_embeddings))
print(semantic_embeddings.shape)

In [ ]:
class LearnedProjector(nn.Module):
    """
    Projects a frozen all-mpnet-base-v2 sentence embedding (768-dim)
    down to a smaller learned representation via a linear layer.

    Note: sentence-transformers' mpnet output is L2-normalized by default
    (normalize_embeddings=True), so no extra normalization is applied here
    on the input side.

    Usage:
        projector = LearnedProjector(in_dim=768, out_dim=64)
        z = projector(x)  # x: (batch, 768) -> z: (batch, 64)
    """
    def __init__(self, in_dim: int = 768, out_dim: int = 64,
                 hidden_dim: int | None = None, dropout: float = 0.1):
        super().__init__()

        if hidden_dim is None:
            self.net = nn.Sequential(
                nn.Linear(in_dim, out_dim),
                nn.LayerNorm(out_dim),
            )
        else:
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, out_dim),
                nn.LayerNorm(out_dim),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

projector = LearnedProjector(in_dim=768, out_dim=64)
semantic_projected = projector(torch.from_numpy(semantic_embeddings))
print(semantic_projected.shape)
print(semantic_projected)

## Fusion Network

The input dimensions are placeholders for now.

In [ ]:
class FusionNetwork(nn.Module):

    def __init__(self):
        super().__init__()

        # Text branch
        self.text_branch = nn.Sequential(
            nn.Linear(384, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )

        # Image branch
        self.image_branch = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )

        # Tabular branch
        self.tabular_branch = nn.Sequential(
            nn.Linear(50, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        # Fusion portion
        self.fusion = nn.Sequential(
            nn.Linear(64 + 64 + 32, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, text, image, tabular):

        text_features = self.text_branch(text)

        image_features = self.image_branch(image)

        tabular_features = self.tabular_branch(tabular)

        combined = torch.cat(
            [text_features, image_features, tabular_features],
            dim=1
        )

        output = self.fusion(combined)

        return output

In [ ]:
X_text = semantic_projected
X_image = []
X_other_pre = df.drop(columns=['mal_id', 'title', 'source', 'producers', 'genres', 'studios', 'demographics', 'themes', 'score_z', 'wc_z', 'favorites_z', 'drop_rate_z', 'forum_z', '__index_level_0__'])
X_other = torch.from_numpy(X_other_pre.to_numpy())
y_score = df['score_z']

In [ ]:
# Train-test split

(
    text_train,
    text_test,
    image_train,
    image_test,
    other_train,
    other_test,
    score_train,
    score_test
) = train_test_split(
    X_text,
    X_image,
    X_other,
    y_score,
    test_size=0.10,
    random_state=42
)

# Train-validation split

(
    text_train,
    text_val,
    image_train,
    image_val,
    other_train,
    other_val,
    score_train,
    score_val
) = train_test_split(
    text_train,
    image_train,
    other_train,
    score_train,
    test_size=0.10,
    random_state=42
)

In [ ]:
model = FusionNetwork()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

train_dataset = TensorDataset(
    text_train,
    image_train,
    other_train,
    score_train
)

val_dataset = TensorDataset(
    text_val,
    image_val,
    other_val,
    score_val
)

test_dataset = TensorDataset(
    text_test,
    image_test,
    other_test,
    score_test
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 50

for epoch in range(epochs):

    model.train()

    train_loss = 0

    for text, image, other, target in train_loader:
        text = text.to(device)
        image = image.to(device)
        other = other.to(device)
        target = target.to(device)

        # Forward pass
        prediction = model(
            text,
            image,
            other
        )

        # Calculate error
        loss = criterion(
            prediction,
            target
        )

        # Reset gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0

    with torch.no_grad(): 
        for inputs, targets in val_loader:
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_loader)
    
    print(f"Epoch {epoch}: Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

checkpoint_path = 'best_model.pth'
model.load_state_dict(torch.load(checkpoint_path, map_location=device))

In [ ]:
total_mse = 0
all_predictions = []
all_targets = []

with torch.no_grad():
    for text, image,  in test_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        
        predictions = model(inputs)
        
        total_mse += criterion(predictions, targets).item() * inputs.size(0)

        all_predictions.append(predictions.cpu())
        all_targets.append(targets.cpu())

total_samples = len(test_loader.dataset)
final_rmse = (total_mse / total_samples) ** 0.5

print(f"Test RMSE: {final_rmse:.4f}")

final_predictions = torch.cat(all_predictions, dim=0).numpy()
final_targets = torch.cat(all_targets, dim=0).numpy()